# Notebook 00c · Sentinel-2 manual stack extraction — Hanoi & HCMC

Builds the **stack** composite (`s2_utils.build_composite(..., method='stack')`)
from a manually chosen date set, at the same sample points
`03_Transferability_Vietnam_S2_median.ipynb` already drew for its median run —
so the stack and the median composites are compared on identical points and
train/test split, and only the predictor set changes.

## Why these dates, not notebook 00b's automated selection

00b's threshold ladder picks dates by cloud/valid/coverage score for a
**median** or **percentile** composite, where one poor date is diluted into an
average. A **stack** keeps every date as its own bands, so a poor date is not
averaged away — it is injected as a column. The dates below were chosen by
hand, keeping only dates at or above 85% valid-pixel coverage over the AOI:

| City | Dates | Bands |
|---|---|---|
| Hanoi | 2018-01-19, 2018-10-31, 2018-12-20 | 30 |
| HCMC | 2018-02-12 | 10 |

HCMC has one qualifying date, so its "stack" is a single-date composite with
no temporal information at all. Milan's stack model has 40 bands (4 dates);
neither city's band count matches it, so a Milan-trained stack classifier
cannot be applied zero-shot to either — see `03b`'s header.

## Pipeline

| Cell | Does |
|---|---|
| 1 | Config |
| 2 | Load `03`'s train/test sample points, tagged by split |
| 3 | Build each city's AOI |
| 4 | Build the stack composite, extract bands at the sample points |
| 5 | Re-split by the tag, save under `output/transfer_vietnam/{city}/stack_manual/` |

> Prerequisite: run `03_Transferability_Vietnam_S2_median.ipynb` first — this
> notebook reads its `samples_{city}_{train,test}.gpkg` outputs directly.

## Cell 1 · Imports & configuration

In [1]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee

init_gee()

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


GEE initialised (project: imd-vietnam).


'imd-vietnam'

In [2]:
# PROJ fix (mirrors notebooks 00b/02/03) -- must precede geopandas/rasterio import
import os
import pyproj
os.environ['PROJ_LIB']  = pyproj.datadir.get_data_dir()
os.environ['PROJ_DATA'] = pyproj.datadir.get_data_dir()

import json
import time
import warnings

import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

from s2_utils import build_composite, water_mask

warnings.filterwarnings('ignore')

# -- Source --------------------------------------------------------------------
S2_COLLECTION    = 'COPERNICUS/S2_SR_HARMONIZED'
COMPOSITE_METHOD = 'stack'

# -- Manually chosen dates ------------------------------------------------------
# Not 00b's threshold ladder -- see header. A stack keeps every date as its own
# bands, so a weak date is not averaged away the way it would be under median
# or percentile; each date below clears 85% valid-pixel coverage over its AOI.
CITIES = {
    'Hanoi': {
        'centre_lonlat': (105.85, 21.03),
        'utm_crs':       'EPSG:32648',
        'dates':         ['2018-01-19', '2018-10-31', '2018-12-20'],
    },
    'HCMC': {
        'centre_lonlat': (106.70, 10.78),
        'utm_crs':       'EPSG:32648',
        'dates':         ['2018-02-12'],
    },
}
BBOX_HALF = 15000   # 30 km x 30 km -- identical to notebooks 00b/02/03

# -- Paths -----------------------------------------------------------------------
# Reuses 03's own train/test split points, not a fresh GHSL sample -- so the
# stack and median composites are compared on identical ground with only the
# predictor set changing.
def median_dir(city):
    return f'./output/transfer_vietnam/{city.lower()}/median'

def output_dir(city):
    d = f'./output/transfer_vietnam/{city.lower()}/stack_manual'
    os.makedirs(d, exist_ok=True)
    return d

print(f'{S2_COLLECTION} | manual stack | {list(CITIES)}')
for city, cfg in CITIES.items():
    print(f'  {city:6s} {len(cfg["dates"])} date(s): {cfg["dates"]} '
          f'-> {10 * len(cfg["dates"])} bands')

COPERNICUS/S2_SR_HARMONIZED | manual stack | ['Hanoi', 'HCMC']
  Hanoi  3 date(s): ['2018-01-19', '2018-10-31', '2018-12-20'] -> 30 bands
  HCMC   1 date(s): ['2018-02-12'] -> 10 bands


---
## Cell 2 · Load the median run's sample points

In [3]:
samples_in = {}
for city in CITIES:
    d = median_dir(city)
    train = gpd.read_file(f'{d}/samples_{city}_train.gpkg')
    test  = gpd.read_file(f'{d}/samples_{city}_test.gpkg')
    train['split'] = 'train'
    test['split']  = 'test'
    pts = pd.concat([train, test], ignore_index=True)
    pts = gpd.GeoDataFrame(pts, geometry='geometry', crs=train.crs)
    samples_in[city] = pts
    print(f'{city:6s} {len(train)} train + {len(test)} test = {len(pts)} points '
          f'loaded from {d}')

Hanoi  2387 train + 895 test = 3282 points loaded from ./output/transfer_vietnam/hanoi/median


HCMC   2362 train + 887 test = 3249 points loaded from ./output/transfer_vietnam/hcmc/median


---
## Cell 3 · Build each city's AOI

In [4]:
def build_aoi(cfg):
    # 30 km x 30 km ee.Geometry around a city centre, built in UTM.
    lon, lat = cfg['centre_lonlat']
    centre = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([lon], [lat]), crs='EPSG:4326'
    ).to_crs(cfg['utm_crs'])
    cx, cy = centre.geometry.x[0], centre.geometry.y[0]
    bbox_utm = box(cx - BBOX_HALF, cy - BBOX_HALF,
                   cx + BBOX_HALF, cy + BBOX_HALF)
    bbox_4326 = gpd.GeoSeries([bbox_utm], crs=cfg['utm_crs']).to_crs('EPSG:4326').iloc[0]
    return ee.Geometry.Polygon([[list(c) for c in bbox_4326.exterior.coords]])


for city, cfg in CITIES.items():
    cfg['aoi_geom'] = build_aoi(cfg)
    area = cfg['aoi_geom'].area(1).getInfo()
    print(f'{city:6s} AOI area: {area / 1e6:,.0f} km2')

Hanoi  AOI area: 903 km2


HCMC   AOI area: 904 km2


---
## Cell 4 · Build the stack composite, extract bands at the sample points

In [5]:
samples_out = {}
band_names_by_city = {}

for city, cfg in CITIES.items():
    print(f'\n{city}:')
    non_water = water_mask(cfg['aoi_geom'])
    img, proj, band_names = build_composite(
        S2_COLLECTION, cfg['dates'], cfg['aoi_geom'],
        non_water=non_water, method=COMPOSITE_METHOD)
    band_names_by_city[city] = band_names
    print(f'  composite: {len(band_names)} bands ({band_names[0]} ... {band_names[-1]})')

    pts = samples_in[city].reset_index(drop=True)
    fc_pts = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([row.geometry.x, row.geometry.y]),
                   {'IMD': float(row['IMD']), 'IMD_class': int(row['IMD_class']),
                    'split': row['split'], 'pid': int(i)})
        for i, row in pts.iterrows()
    ])

    sampled = img.sampleRegions(
        collection=fc_pts, properties=['IMD', 'IMD_class', 'split', 'pid'],
        scale=10, projection=proj, tileScale=4, geometries=True)

    t0 = time.time()
    gdf = geemap.ee_to_gdf(sampled)
    print(f'  extracted: {len(gdf)} / {len(pts)} points ({time.time() - t0:.1f}s)')

    # Masked wherever no selected date mosaicked over that pixel -- a stack has
    # no cross-date fallback for a point outside every date's footprint, unlike
    # the per-pixel median fallback build_composite applies inside each date.
    n_before = len(gdf)
    gdf = gdf.dropna(subset=band_names).reset_index(drop=True)
    retention = 100 * len(gdf) / max(n_before, 1)
    print(f'  after NaN drop: {len(gdf)} ({retention:.1f}% retained)')

    samples_out[city] = gdf


Hanoi:
  composite: 30 bands (B2_d00 ... B12_d02)


  extracted: 3282 / 3282 points (107.2s)
  after NaN drop: 3282 (100.0% retained)

HCMC:
  composite: 10 bands (B2_d00 ... B12_d00)


  extracted: 3249 / 3249 points (66.2s)
  after NaN drop: 3249 (100.0% retained)


---
## Cell 5 · Re-split and save

In [6]:
for city, gdf in samples_out.items():
    out_dir = output_dir(city)
    band_names = band_names_by_city[city]

    # sampleRegions does not promise to preserve input order, so split by the
    # 'split' tag carried through as a property -- not by position.
    train = gdf[gdf['split'] == 'train'].drop(columns=['split']).reset_index(drop=True)
    test  = gdf[gdf['split'] == 'test' ].drop(columns=['split']).reset_index(drop=True)

    gdf.to_file(f'{out_dir}/samples_{city}_stack_all.gpkg', driver='GPKG')
    train.to_file(f'{out_dir}/samples_{city}_train.gpkg', driver='GPKG')
    test.to_file(f'{out_dir}/samples_{city}_test.gpkg', driver='GPKG')

    meta = {
        'method':      COMPOSITE_METHOD,
        'dates':       CITIES[city]['dates'],
        'band_names':  band_names,
        'n_train':     len(train),
        'n_test':      len(test),
    }
    with open(f'{out_dir}/s2_stack_manual_metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'{city:6s} saved {len(train)} train + {len(test)} test '
          f'({len(band_names)} bands) -> {out_dir}')

print('\nDone. Run 03b_Transferability_Vietnam_S2_stack_manual.ipynb next.')

Hanoi  saved 2387 train + 895 test (30 bands) -> ./output/transfer_vietnam/hanoi/stack_manual


HCMC   saved 2362 train + 887 test (10 bands) -> ./output/transfer_vietnam/hcmc/stack_manual

Done. Run 03b_Transferability_Vietnam_S2_stack_manual.ipynb next.
